<a href="https://colab.research.google.com/github/kennedy336/Machine-Learning-and-Deep-learning-mini-Projects/blob/main/LLM%20%2B%20RAG_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# setup_bedrock.py
from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage, SystemMessage

# Initialiser le LLM Nova Lite
llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
    model_kwargs={
        "max_tokens": 1024,
        "temperature": 0.1
    }
)

# Appel simple
msgs = [
    SystemMessage("Expert forex"),
    HumanMessage("Analyse EURUSD")
]
resp = llm.invoke(msgs)
print(resp.content)

# Streaming token par token
for chunk in llm.stream(msgs):
    print(chunk.content, end="", flush=True)

In [ ]:
# chain_lcel.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

# Template avec variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "Expert {domain}."),
    ("human", "{question}")
])

# Composition par pipe
chain = prompt | llm | StrOutputParser()

# Execution
result = chain.invoke({
    "domain": "forex",
    "question": "Signal EURUSD?"
})
print(result)

In [ ]:
# json_output.py
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_aws import ChatBedrock
from pydantic import BaseModel, Field

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

# Schema de sortie
class Signal(BaseModel):
    pair: str = Field(description="EURUSD")
    direction: str = Field(description="BUY/SELL")
    confidence: float = Field(ge=0, le=1)
    reason: str

parser = JsonOutputParser(pydantic_object=Signal)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Expert forex. {format_instructions}"),
    ("human", "Analyse {pair}")
])

chain = (prompt.partial(format_instructions=parser.get_format_instructions())
| llm
    | parser
)

result = chain.invoke({"pair": "EURUSD"})
print(result)

In [ ]:
# memory_chat.py
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Expert forex."),
    MessagesPlaceholder('history'),
    ("human", "{input}")
])

chain = prompt | llm

store = {}  # session_id -> history

def get_hist(sid):
    if sid not in store:
        store[sid] = InMemoryChatMessageHistory()
    return store[sid]

chat = RunnableWithMessageHistory(
    chain,
    get_hist,
    input_messages_key="input",
    history_messages_key="history"
)

# Utilisation
response = chat.invoke(
    {"input": "Que penses-tu de l'EURUSD ?"},
    config={"configurable": {"session_id": "user_123"}}
)
print(response.content)

In [ ]:
# agent_react.py
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import tool
from langchain import hub
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

@tool
def get_forex_rate(pair: str) -> str:
    #Taux forex actuel.
    import yfinance as yf
    d = yf.download(f'{pair}=X', period='1d')
    return f"{pair}: {d['Close'].iloc[-1]:.4f}"

@tool
def calc(expression: str) -> str:
    #Calcule une expression.
    return str(eval(expression))

tools = [get_forex_rate, calc]
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)

executor = AgentExecutor(
    agent=agent, tools=tools,
    verbose=True, max_iterations=5
)

executor.invoke({"input": "Quel est le taux de l'EURUSD et multiplie-le par 1.1 ?"})

In [ ]:
# rag_llamaindex.py
from llama_index.llms.bedrock import Bedrock
from llama_index.embeddings.bedrock import BedrockEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings

# Config globale
Settings.llm = Bedrock(
    model="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)
Settings.embed_model = BedrockEmbedding(
    model_name="amazon.titan-embed-text-v2:0"
)

# 3 lignes = RAG complet
docs = SimpleDirectoryReader("./docs/").load_data()
index = VectorStoreIndex.from_documents(docs)
engine = index.as_query_engine(similarity_top_k=5)

# Query
answer = engine.query("Signal EURUSD?")
print(answer.response)
for n in answer.source_nodes:
    print(n.node.text[:100], "...")

In [ ]:
# embed_titan.py
import boto3, json, numpy as np

client = boto3.client(
    "bedrock-runtime",
    region_name="us-east-1"
)

def embed(text: str) -> np.ndarray:
    #Texte -> vecteur 1024d.
    r = client.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({
            'inputText': text,
            'dimensions': 1024,
            'normalize': True
        }))
    data = json.loads(r['body'].read())
    return np.array(data['embedding'], dtype='float32')

# Test
v = embed("EURUSD monte fort")
print(f"Shape: {v.shape}")  # (1024,)
print(f"Norme: {np.linalg.norm(v):.4f}")  # ~1.0

In [ ]:
# faiss_index.py
import faiss, numpy as np

# Corpus de news forex
docs = ["EURUSD +0.5% apres NFP solide",
    "BCE maintient taux a 4.25%",
    "DXY recule: dollar faiblit",
    "Fed: pause sur les taux US",]

# Vectoriser + indexer (suppose la fonction embed() définie à l'étape 7)
vecs = np.stack([embed(d) for d in docs])
index = faiss.IndexFlatIP(1024)
index.add(vecs)
print(f"Index: {index.ntotal} vecteurs")

# Recherche semantique
q = embed("hausse euro dollar")
D, I = index.search(q.reshape(1, -1), k=3)
for score, idx in zip(D[0], I[0]):
    print(f"  [{score:.3f}] {docs[idx]}")

In [ ]:
# rag_complete.py
# (Apres avoir indexé les docs et initialisé FAISS et embed() des steps 7 & 8)
import boto3

br = boto3.client('bedrock-runtime', region_name="us-east-1")

def rag_query(question):
    # 1. Retrieval
    q = embed(question).reshape(1, -1)
    D, I = index.search(q, k=3)
    ctx = "\n".join(docs[i] for i in I[0])

    # 2. Generation augmentee
    prompt = f"""Contexte:\n{ctx}\n\nQuestion: {question}\nReponds UNIQUEMENT avec le contexte."""

    resp = br.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=[{'role': 'user', 'content': [{'text': prompt}]}]
    )
    return resp['output']['message']['content'][0]['text']

print(rag_query("Quelle est la situation sur l'EURUSD ?"))

In [ ]:
# eval_ragas.py
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_recall, context_precision
)
from datasets import Dataset

eval_data = {
    "question": ["Signal EURUSD?", "BCE monte les taux?"],
    "answer": ["Haussier +0.5%", "Non, maintenus 4.25%"],
    "contexts": [
        ["EURUSD +0.5% apres NFP"],
        ["BCE: taux maintenus"]
    ],
    "ground_truth": ["EURUSD haussier", "Taux stables"]
}

ds = Dataset.from_dict(eval_data)

result = evaluate(ds, metrics=[
    faithfulness, answer_relevancy,
    context_recall, context_precision
])
print(result)

In [ ]:
# guardrails_check.py
import boto3

br = boto3.client('bedrock-runtime', region_name="us-east-1")

def check(text, direction):
    """Verifie input ou output."""
    r = br.apply_guardrail(
        guardrailIdentifier="your-id",
        guardrailVersion="1",
        source=direction,
        content=[{
            "text": {"text": text}
        }]
    )
    return r

# Usage dans le pipeline RAG
def rag_secure(question):
    gi = check(question, "INPUT")
    if gi['action'] == 'GUARDRAIL_INTERVENED':
        return 'Question bloquee'

    # Appel de la fonction de RAG (définie à l'étape 9)
    answer = rag_query(question)

    go = check(answer, "OUTPUT")
    if go['action'] == 'GUARDRAIL_INTERVENED':
        return 'Reponse bloquee pour des raisons de securite'

    return answer